# 🏭 Factory 5.0 — MAPPO Dual-Arm Training
**Run on**: Google Colab (T4 GPU free tier)

**Trains**: Multi-Agent PPO for UR5e + Franka box packing pipeline

**Architecture**: CTDE — Centralized Critic + 2 Decentralized Actors

---
### Steps:
1. Upload `dual_arm_packing_env.py` and `mappo_agent.py` to Colab
2. Run all cells
3. Download trained model → place in `factory50_ws/src/factory50_marl/models/`
4. Deploy via `rl_inference_node.py` in ROS2

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install gymnasium torch torchvision wandb --quiet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Upload your env and agent files ───────────────────────────────
from google.colab import files
print('Upload dual_arm_packing_env.py and mappo_agent.py')
uploaded = files.upload()

In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────
import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 4: Stage 1 — Solo training (no human, Arm 0 only) ───────────────
from dual_arm_packing_env import DualArmPackingEnv
from mappo_agent import MAPPOTrainer
import numpy as np

print('=== STAGE 1: Solo Training (no human) ===')
env_stage1 = DualArmPackingEnv(use_human=False)
trainer    = MAPPOTrainer(env_stage1, config={
    'lr_actor':     3e-4,
    'rollout_steps': 2048,
    'n_epochs':     10,
})
trainer.train(total_steps=1_000_000, save_path='/content/models/stage1')
print('Stage 1 complete ✅')

In [ ]:
# ── Cell 5: Stage 2 — Static human ───────────────────────────────────────
print('=== STAGE 2: Static Human ===')
env_stage2 = DualArmPackingEnv(use_human=True)
# Fix human position (static)
env_stage2.human_pos[:] = [0.0, 1.8, 0.0]   # human standing 1.8m away

trainer_s2 = MAPPOTrainer(env_stage2)
trainer_s2.load('/content/models/stage1/best_model.pt')  # warm-start
trainer_s2.train(total_steps=500_000, save_path='/content/models/stage2')
print('Stage 2 complete ✅')

In [ ]:
# ── Cell 6: Stage 3 — Full Factory 5.0 (dynamic human) ───────────────────
print('=== STAGE 3: Full Factory 5.0 — Dynamic Human ===')
env_stage3 = DualArmPackingEnv(use_human=True)

trainer_s3 = MAPPOTrainer(env_stage3, config={'entropy_coef': 0.005})
trainer_s3.load('/content/models/stage2/best_model.pt')  # warm-start
trainer_s3.train(total_steps=2_000_000, save_path='/content/models/stage3')
print('Stage 3 complete ✅')

In [ ]:
# ── Cell 7: Evaluate trained model ───────────────────────────────────────
import torch

env_eval  = DualArmPackingEnv(use_human=True, render_mode='human')
trainer_eval = MAPPOTrainer(env_eval)
trainer_eval.load('/content/models/stage3/best_model.pt')

successes  = 0
n_episodes = 100

for ep in range(n_episodes):
    obs, _ = env_eval.reset()
    done   = False
    while not done:
        with torch.no_grad():
            obs0_t = torch.FloatTensor(obs['arm0']).unsqueeze(0)
            obs1_t = torch.FloatTensor(obs['arm1']).unsqueeze(0)
            act0, _, _ = trainer_eval.actor_arm0.get_action(obs0_t)
            act1, _, _ = trainer_eval.actor_arm1.get_action(obs1_t)
        actions = {'arm0': act0.squeeze(0).numpy(),
                   'arm1': act1.squeeze(0).numpy()}
        obs, _, terminated, truncated, info = env_eval.step(actions)
        done = terminated or truncated
        if info.get('box_packed'): successes += 1

print(f'\nSuccess rate: {successes}/{n_episodes} = {successes}%')
print('✅ Evaluation complete — download best_model.pt below')

In [ ]:
# ── Cell 8: Download model ────────────────────────────────────────────────
from google.colab import files
files.download('/content/models/stage3/best_model.pt')
print('Downloaded! Place in: factory50_ws/src/factory50_marl/models/best_model.pt')